In [ ]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
sys.path.insert(0, "..")
from importlib import reload
import matplotlib.pyplot as plt
import equations as eq
reload (eq);
import trajectory_lib as tr
reload (tr);

participant = 'par2'
GH_seq = 'YZY'

# Validaiton simulations also uses OS_model_prediction struct as it includes the polynomials for GH reaction forces vectors
OS_struct = sc.io.loadmat('../Motions/'+participant+'/OS_model_prediction.mat')

act_w = 1
include_activation_dynamics = True
MM,FO,q,u,faux,fr,frstar,kinematical,xdot,first_elips_scale,elips_trans = eq.create_eoms_quat_w_RF(OS_struct,weight = 0,derive = 'numeric',gen_matlab_functions = 0)
clav_pos = 0.4
# Define inclination (around z-axis) and version (around y-axis) angles of the glenoid.
tilt_y = 13
tilt_z = -6.5
w_traj = 200
wGH = 2
simulations = ['shelf_reaching'] #,'driving','drinking','lifting_5kg']
params = [0,1]
w_diff_vel = 1e-3
w_diff_exc = 1e-3
w_diff_faux = 1e-3
thor_hum_only = False
for cur_mot in simulations:
    for cur_param in params:

        if cur_mot =='lifting_5kg':
            MM,FO,q,u,faux,fr,frstar,kinematical,xdot,first_elips_scale,elips_trans = eq.create_eoms_quat_w_RF(OS_struct,weight = 5,derive = 'numeric',gen_matlab_functions = 0)

    
        if cur_param == 0:
            calibrated_params = None
        else:
            calibrated_params = sc.io.loadmat('../Motions/'+participant+'/calibrated_params.mat')

        TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces, mus_forces_objective = eq.polynomials_quat(OS_struct,q,u,calibrated_params = calibrated_params, derive = 'numeric', RC_lim = 1.0)

        eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+sp.Matrix([TE+sp.Matrix(TE_conoid)]).col_join(GH_mus_forces))

        if include_activation_dynamics:
            excitations = []
            act_ode = []
            for i in range(len(activations)):
                excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
                current_mus_ind = int(str(activations[i])[4:-3])
                current_mus = OS_struct['model']['muscles'].item()[0,(current_mus_ind-1)]
                t_act = current_mus['tact'][0,0].item()
                t_deact = current_mus['tdeact'][0,0].item()
                act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i]))
            sp_act_ode = sp.Matrix(act_ode)
            eoms_implicit = eoms_implicit.col_join(sp_act_ode)

        interval_value = 0.04
        file = '../Motions/' + participant + '/' + cur_mot + '/' + cur_mot
        traj_original, omega, num_nodes, time = tr.exp_trajectory_quat(file,interval_value)
        q0_t0 = traj_original[:,0][:4]
        indexes = np.ones(num_nodes)

        traj = tr.exp_trajectory_quat_myobj(traj_original,clav_pos)

        if include_activation_dynamics:
            state_symbols = tuple(q+u+faux+activations)
            specified_symbols = tuple(excitations)
        else:
            state_symbols = tuple(q+u+faux)
            specified_symbols = tuple(activations)

        num_states = len(state_symbols) 
        num_q = len(q)
        num_u = len(u)
        num_faux = len(faux)
        num_inputs = len(specified_symbols)
        t = me.dynamicsymbols._t
        
        objective_traj,objective_traj_jac, objective_SC_t0, objective_SC_t0_jac = eq.objective_traj_quat(num_q,interval_value,clav_pos,True)

        objective_act,objective_act_jac = eq.objective_min_activation(activations,interval_value)
        objective_exc,objective_exc_jac = eq.objective_min_activation(activations,interval_value)
        objective_maxstab, objective_maxstab_jac = eq.objective_max_GH_stab(tilt_y=tilt_y,tilt_z=tilt_z,interval_value=interval_value)

        obj_min_diff,obj_min_diff_jac = eq.objective_state_diff(num_nodes,interval_value)

        indexes_clav_scap = 1
        indexes_hum = 1
    

        def obj(free):
            min_traj = w_traj * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj,indexes_clav_scap,indexes_hum))
            min_SC_t0 = w_traj * np.sum(objective_SC_t0(free[0::num_nodes][:4],q0_t0))

            min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
            min_faux_dif = w_diff_faux * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))))


            min_act = act_w * np.sum(objective_act(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs)))


            min_instab = wGH * np.sum(objective_maxstab(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

            obj = (min_traj + min_vel_dif + min_act + min_instab + min_faux_dif + min_SC_t0) #   

            if include_activation_dynamics:
                min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))))
                obj += (min_exc_dif)

            return obj.item()

        def obj_grad(free):
            grad = np.zeros_like(free)

            grad[:num_q*num_nodes] += w_traj * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj,indexes_clav_scap,indexes_hum))
            grad[0::num_nodes][:4] += w_traj * np.sum(objective_SC_t0_jac(free[0::num_nodes][:4],q0_t0))

            grad[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes] += act_w * np.concatenate(objective_act_jac(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs)))

            grad[num_q*num_nodes:(num_q + num_u)*num_nodes] += w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))
            grad[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes] += w_diff_faux * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))[0,:,:])))

            grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes] += wGH * np.concatenate(objective_maxstab_jac(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

            if include_activation_dynamics:
                grad[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes] += w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))[0,:,:])))

            return grad
        
        print('obj_check', obj(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01))
        print('obj_grad_check', sum(obj_grad(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01)))


        instance_constraints = []
        instance_constraints.append(state_symbols[0].func(0)**2 + state_symbols[1].func(0)**2 + state_symbols[2].func(0)**2 + state_symbols[3].func(0)**2 - 1) # SC
        instance_constraints.append(state_symbols[4].func(0)**2 + state_symbols[5].func(0)**2 + state_symbols[6].func(0)**2 + state_symbols[7].func(0)**2 - 1) # AC
        instance_constraints.append(state_symbols[8].func(0)**2 + state_symbols[9].func(0)**2 + state_symbols[10].func(0)**2 + state_symbols[11].func(0)**2 - 1) # GH

            
        bounds1 = (0.0,1.0)
        bounds = (bounds1,)*len(activations)
        bndrs = dict(zip(activations,bounds))
        if include_activation_dynamics:
            bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
            bndrs.update(bndrs_exc)

                    

        for i in range(num_q):
            if i == 8:
                bndrs.update({q[i]: (min(traj_original[i,:])-0.15, 1.0)})
            else:
                bndrs.update({q[i]: (min(traj_original[i,:])-0.15, max(traj_original[i,:])+0.15)})

        bndrs.update({faux[0]: (-2,0)})


        start = tm.time()
        prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                    num_nodes, interval_value,
                    known_parameter_map={},
                    instance_constraints=instance_constraints,
                    bounds=bndrs,
                    integration_method='midpoint',
                    parallel = False)


        time_to_create = tm.time() - start
        print(time_to_create)

        prob.add_option('limited_memory_max_history', 40)
        initial_guess = np.ones(prob.num_free)*0.0
        time_2_solve_start = tm.time()
        prob.add_option('max_iter',2000)
        initial_guess[:13*num_nodes] = traj_original.flatten()
        initial_guess[num_q * num_nodes : (num_q + num_u) * num_nodes] = omega.flatten()
        initial_guess[(num_q + num_u)*num_nodes:(num_q + num_u + 1)*num_nodes] = -0.75


        solution, info = prob.solve(initial_guess)
        time_2_solve = tm.time() - time_2_solve_start
        print(info['status_msg'])
        print(info['obj_val'])
        act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
        objective_value = prob.obj_value
        print('Objective activations: ', act_obj)

        file_name = '../Motions/'+participant+'/'+cur_mot+'/res_' + cur_mot + '_' + str(cur_param)+'.mat'

        tr.sol2struct(solution,activations,num_q,num_u,num_faux,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

        file_name_mot = '../Motions/'+participant+'/'+cur_mot+'/res_' + cur_mot + '_' + str(cur_param)+'.mot'
        tr.sol2mot_quat(solution, num_nodes, len(q), time, file_name_mot, GH_seq)

infra1  .. 272.0
infra2  .. 275.0
infra3  .. 194.0
infra4  .. 207.0
infra5  .. 302.0
infra6  .. 182.0
supra1  .. 120.0
supra2  .. 113.0
supra3  .. 258.0
supra4  .. 130.0
[0.1280213209318535, 0.09977871970276542, 0.09473223609756792, 0.08428512968680808, 0.0818061552842549, 0.0790615764814282, 0.07684820647914856, 0.06817179607021243, 0.089508682892188, 0.11190798731525782, 0.11996465412355564, 0.13696333574106312, 0.14873846415319078, 0.1397079145438899, 0.11527230971872285, 0.06772912206975651, 0.09614879289902686, 0.09517491009802384, 0.12226655892592643, 0.11385575291726387, 0.12191241972556167, 0.1380257533421574, 0.13873403174288687, 0.13899963614316038, 0.09738822274168242, 0.11748559310816087, 0.11704286818258923, 0.11412124455528062, 0.11146519949003068, 0.1080123389766735, 0.1003097998889621, 0.07481180430570758, 0.07082777389651698, 0.05843292559889128, 0.07835319856463996, 0.09828496299337719, 0.08260005343976276, 0.07230833269217109, 0.057742059445250064, 0.0568097682953499